# Input Grad Steering — GRU · H=256

**Question.** Same as the transformer notebook (`input_grad_steering_transformer.ipynb`): can the gradient of a
frozen linear position probe, backpropagated **through the GRU (encoder + recurrence) to the input observation**,
produce a *semantically* edited observation — or an adversarial perturbation the dynamics ignore? Unlike the
transformer, the GRU's carried state is `h`, not the observation buffer: here the steered frame acts on the state
**once**, through the update `h_ef = GRU(enc(obs), h_prev)`, and must survive recurrently.

**Data / model provenance.** Model `H256` = **GRU · H=256 (reference)**, checkpoint
`runs/controls/H256/best_model.pt`, row copied from `../controls/CONTROL_RUNS.md`. Dataset
`datasets/4_fixed_refl_inview` (2 objects, T=40, R=128, obs noise 0.2, edit frame `ef`=20). N=64 edits, K=15
rollout steps. §4 metrics from `scripts/editability_metrics.py`; probes from `pim.extractors.fit_readability_probes`.

## Definitions

| term / metric | definition | units | better |
|---|---|---|---|
| **linear / MLP probe R²** | held-out R² of the standard readability probes on `h` (`fit_readability_probes`: linear lstsq + 2×256 ReLU MLP, 80/20 by-**sequence** split), target = positions of both objects (4 dims) | — | ↑ |
| **Input Grad Steering (n=1, λ)** | Adam (300 steps, lr 0.02) on δ added to `obs[ef−1]` (the last teacher-forced frame), minimizing `‖A·h_ef(δ) + b − target‖² + λ‖δ‖²`, input clamped to [0,1], where `h_ef(δ) = GRU(enc((obs[ef−1]+δ)), h_prev)` and `(A, b)` is the frozen standard linear probe on `h`; target = edited object → teleport target, other object → its true `ef` position | — | — |
| **Render write @1 (oracle)** | same write surface, oracle content: feed the clean render of the edited world (`gt_edited`) instead of `obs[ef−1]` | — | — |
| **Δ_true (true edit direction)** | `gt_edited − obs[ef−1]` (impurities as stated in `README.md`: base-frame noise + one frame of motion offset) | intensity | — |
| **cos(δ, Δ_true) / angle** | per-sample cosine (and angle, degrees) vs Δ_true; chance = shuffled-pair control | — / ° | ↑ / ↓ |
| **probe residual** | `‖A·h + b − target‖` after steering, mean over samples | sim units | ↓ |
| **Edit Index / zone RMSEs / GT-traj RMSE / fidelity ratio** | canonical §4 set, exactly as defined in the transformer notebook and `../METRICS_AND_EDITORS.md` §4 | — | see registry |

All observation-space errors are scored against the **clean** render. Rollout step 0 decodes frame `ef`.
Timing note: steering perturbs the **last warm-up frame** (`obs[ef−1]`), so the steered arm sees exactly as many
frames as the unsteered one — no one-frame lead (unlike First Obs. TF).

In [ ]:
# [1] Setup + standard readability probes on h.
import os, sys
sys.path.insert(0, "../../../..")
sys.path.insert(0, "../../../../scripts")
import numpy as np, torch, h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown

from pim.world_models.loader import load_checkpoint, load_dataset
from pim.extractors import fit_readability_probes
from pim.figures.theme import style_ax
from editability_metrics import build_edit_zones, edit_scorecard, fidelity_ratio

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ = 2
N_EDIT, K, N_PROBE = 64, 15, 800
STEER_STEPS, STEER_LR = 300, 0.02
LAM_MAIN = 0.1
LAMBDAS = [0.0, 0.01, 0.1, 1.0]
OUT = "/tmp/input_grad_steering_gru"; os.makedirs(OUT, exist_ok=True)

MODEL_LABEL = "GRU · H=256"
model, info = load_checkpoint("../../../../runs/controls/H256/best_model.pt", device=DEVICE)
bundle = load_dataset("../../../../datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
ef, R = edits.edit_frame, edits.obs_res
sim = test.config["dataset"]["sim"]
print(f"model {MODEL_LABEL} | epoch {info.epoch} val {info.val_loss:.5f} | H={model.hidden_size} | "
      f"ef={ef} R={R} | N_EDIT={N_EDIT} K={K} device={DEVICE}")

obs_probe = torch.from_numpy(test.obs[:N_PROBE]).float().to(DEVICE)
Tm1 = test.obs.shape[1] - 1
P_tgt = test.positions[:N_PROBE, :Tm1].reshape(N_PROBE, Tm1, N_OBJ * 2).astype(np.float32)
vis = test.is_visible[:N_PROBE, :Tm1, :N_OBJ].all(axis=2)
with torch.no_grad():
    Hs = model.get_hidden_states(obs_probe).cpu().numpy()
PROBE = fit_readability_probes(Hs, P_tgt, mask=vis, device=DEVICE)
display(Markdown(f"**Held-out position R² on `h`** (standard probes, {PROBE['n_train_seq']}/"
                 f"{PROBE['n_heldout_seq']} train/held-out sequences): linear **{PROBE['linear_r2']:.3f}**, "
                 f"MLP **{PROBE['mlp_r2']:.3f}**"))

In [ ]:
# [2] §4 machinery: edit zones, warm-up states, references (unsteered + oracle render write).
N = min(N_EDIT, edits.n_samples)
oe = edits.edit_object[:N].astype(int)
with h5py.File(edits.h5_path, "r") as f:
    pre_vel = f["velocities"][:N, ef - 1, :N_OBJ, :].astype(np.float32)
pre_pos = edits.positions[:N, ef - 1, :N_OBJ, :].astype(np.float32)
tgt_pos = edits.positions[:N, ef, :N_OBJ, :].astype(np.float32)
target4 = torch.from_numpy(tgt_pos.reshape(N, N_OBJ * 2)).float().to(DEVICE)
gt_roll = edits.clean_obs[:N, ef:ef + K, :].astype(np.float32)

ZONES = build_edit_zones(pre_pos=pre_pos, tgt_pos=tgt_pos, pre_vel=pre_vel,
                         edit_object=oe, sim=sim, n_obj=N_OBJ,
                         traj_pos=edits.positions[:N, ef:ef + K, :N_OBJ, :].astype(np.float32),
                         gt_edited_traj=gt_roll)
teleport = ZONES.teleport
gt_edited_t = torch.from_numpy(ZONES.gt_edited).float().to(DEVICE)

obs_e = torch.from_numpy(edits.obs[:N]).float().to(DEVICE)
base_frame = obs_e[:, ef - 1]                              # obs[ef-1], the steered frame
delta_true = ZONES.gt_edited - base_frame.cpu().numpy()

@torch.no_grad()
def warm_state(upto):
    state = None
    for t in range(upto):
        _, state = model.step(obs_e[:, t], state)
    return state
state_prev = warm_state(ef - 1)                            # h after obs[0..ef-2]

@torch.no_grad()
def rollout_from(state, k=K):
    x = model.decode(state); out = [x]; s = state
    for _ in range(k - 1):
        x, s = model.step(x, s)
        out.append(x)
    return torch.stack(out, 1).cpu().numpy()

CARDS, ROLLS = {}, {}
with torch.no_grad():
    _, state_unsteered = model.step(base_frame, state_prev)          # the normal warm-up
    _, state_oracle = model.step(gt_edited_t, state_prev)            # oracle content, same surface
ROLLS["Unsteered"] = rollout_from(state_unsteered)
CARDS["Unsteered"] = edit_scorecard(ROLLS["Unsteered"], ZONES, gt_roll)
ROLLS["Render write @1 (oracle)"] = rollout_from(state_oracle)
CARDS["Render write @1 (oracle)"] = edit_scorecard(ROLLS["Render write @1 (oracle)"], ZONES, gt_roll)
print(f"N={N} edits | mean teleport {teleport.mean():.2f} sim-units | "
      f"unsteered Edit Index {CARDS['Unsteered']['edit_index']:+.2f} | "
      f"oracle render-write Edit Index {CARDS['Render write @1 (oracle)']['edit_index']:+.2f}")

In [ ]:
# [3] The editor: Input Grad Steering (n=1, λ) — Adam on δ over obs[ef−1], through encoder + GRU cell.
A_t = torch.from_numpy(PROBE["A"]).to(DEVICE)   # (4, H)
b_t = torch.from_numpy(PROBE["b"]).to(DEVICE)

def h_from(frame):
    """Differentiable h after feeding `frame` on top of the frozen pre-edit state."""
    x = model._enc(frame).unsqueeze(1)
    h_out, h_next = model.gru(x, state_prev)
    return h_out[:, 0], h_next

def input_grad_steer(lam, steps=STEER_STEPS, lr=STEER_LR):
    delta = torch.zeros(N, R, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)
    first_grad = None
    # cudnn RNN backward is train-mode-only; disable cudnn here so the frozen eval-mode GRU is differentiable
    with torch.backends.cudnn.flags(enabled=False):
        for it in range(steps):
            h, _ = h_from((base_frame + delta).clamp(0, 1))
            probe_loss = ((h @ A_t.T + b_t - target4) ** 2).sum(-1).mean()
            loss = probe_loss + lam * (delta ** 2).sum(-1).mean()
            opt.zero_grad(); loss.backward()
            if it == 0:
                first_grad = -delta.grad.detach().clone()
            opt.step()
    with torch.no_grad():
        steered = (base_frame + delta).clamp(0, 1)
        h, state = h_from(steered)
        resid_after = (h @ A_t.T + b_t - target4).norm(dim=-1).mean().item()
        h0, _ = h_from(base_frame)
        resid_before = (h0 @ A_t.T + b_t - target4).norm(dim=-1).mean().item()
    return dict(delta=(steered - base_frame).cpu().numpy(), first_grad=first_grad.cpu().numpy(),
                state=state, resid_before=resid_before, resid_after=resid_after)

ARMS = {}
for lam in LAMBDAS:
    name = f"Input Grad (n=1, λ={lam})"
    ARMS[name] = input_grad_steer(lam)
    ROLLS[name] = rollout_from(ARMS[name]["state"])
    CARDS[name] = edit_scorecard(ROLLS[name], ZONES, gt_roll)
    a = ARMS[name]
    print(f"{name:28s} probe residual {a['resid_before']:.3f} → {a['resid_after']:.3f} sim-units | "
          f"‖δ‖ {np.linalg.norm(a['delta'], axis=-1).mean():.3f} "
          f"(ref ‖Δ_true‖ {np.linalg.norm(delta_true, axis=-1).mean():.3f})")

In [ ]:
# [4] Fig 2 — what the gradient does to the observation. Rows: 3 large-teleport samples; columns: λ.
SHOW_LAM = [0.0, 0.1, 1.0]
SAMPLES = list(np.argsort(teleport * (ZONES.ghost.sum(1) >= 3))[::-1][:3])
x_axis = np.arange(R)

fig, axes = plt.subplots(len(SAMPLES), len(SHOW_LAM), figsize=(4.6 * len(SHOW_LAM), 2.9 * len(SAMPLES)),
                         sharex=True, sharey=True)
for r, smp in enumerate(SAMPLES):
    for c, lam in enumerate(SHOW_LAM):
        ax = axes[r, c]
        arm = ARMS[f"Input Grad (n=1, λ={lam})"]
        steered = base_frame[smp].cpu().numpy() + arm["delta"][smp]
        ax.plot(x_axis, base_frame[smp].cpu().numpy(), color="#999999", lw=1.0,
                label="original obs[ef−1] (noisy)")
        ax.plot(x_axis, ZONES.gt_edited[smp], color="#009E73", lw=1.4, ls="--",
                label="clean edited-world render (target look)")
        ax.plot(x_axis, steered, color="#D55E00", lw=1.4, label="steered observation")
        ax.fill_between(x_axis, 0, 1, where=ZONES.target[smp], color="#009E73", alpha=0.08)
        ax.fill_between(x_axis, 0, 1, where=ZONES.ghost[smp], color="#FF5252", alpha=0.08)
        ax.set_ylim(-0.05, 1.1)
        if r == 0:
            ax.set_title(f"λ = {lam}", fontsize=10)
        if c == 0:
            ax.set_ylabel(f"sample {smp}\n(teleport {teleport[smp]:.1f})\nintensity", fontsize=9)
        if r == len(SAMPLES) - 1:
            ax.set_xlabel("ray")
        style_ax(ax)
handles, labels = axes[0, 0].get_legend_handles_labels()
handles += [Line2D([0], [0], color="#009E73", lw=6, alpha=0.2), Line2D([0], [0], color="#FF5252", lw=6, alpha=0.2)]
labels += ["target rays (shaded)", "ghost rays (shaded)"]
fig.legend(handles, labels, loc="upper center", ncol=5, fontsize=9, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.suptitle(f"Fig 2 — steered observation vs the true edited-world render, by ridge weight λ "
             f"({MODEL_LABEL}, Input Grad n=1)", y=1.06, fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{OUT}/fig2_steered_obs.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# [5] Fig 3 — cos(δ, Δ_true) + angle by λ, with shuffled-pair chance; then the §4 scorecard + Fig 4.
def cos_rows(vecs):
    num = (vecs * delta_true).sum(-1)
    den = np.linalg.norm(vecs, axis=-1) * np.linalg.norm(delta_true, axis=-1) + 1e-12
    c = num / den
    return float(c.mean()), float(np.degrees(np.arccos(np.clip(c, -1, 1))).mean())

shuf = np.roll(ARMS[f"Input Grad (n=1, λ={LAM_MAIN})"]["delta"], 1, axis=0)
cos_chance, _ = cos_rows(shuf)
stats = []
for lam in LAMBDAS:
    name = f"Input Grad (n=1, λ={lam})"
    c_opt, a_opt = cos_rows(ARMS[name]["delta"])
    c_g, a_g = cos_rows(ARMS[name]["first_grad"])
    stats.append((f"λ = {lam}", c_opt, a_opt, c_g, a_g))
rows = "\n".join(f"| {n} | {c:+.3f} | {a:.0f}° | {cg:+.3f} | {ag:.0f}° |" for n, c, a, cg, ag in stats)
display(Markdown("**Alignment of the steering perturbation with the true edit direction** "
                 f"(shuffled-pair chance cosine = {cos_chance:+.3f}; first gradient is λ-independent up to sign)\n\n"
                 "| arm | cos(δ*, Δ_true) | angle | cos(first grad, Δ_true) | angle |\n|---|---|---|---|---|\n" + rows))

fig, ax = plt.subplots(figsize=(7.5, 3.0))
y = np.arange(len(LAMBDAS))
ax.barh(y - 0.18, [s[1] for s in stats], height=0.36, color="#0072B2", label="converged δ*")
ax.barh(y + 0.18, [s[3] for s in stats], height=0.36, color="#E69F00", label="raw first gradient")
ax.axvline(cos_chance, color="#555555", ls=":", lw=1.4, label="shuffled-pair chance")
ax.set_yticks(y); ax.set_yticklabels([s[0] for s in stats]); ax.invert_yaxis()
ax.set_xlabel("cosine with the true edit direction Δ_true"); ax.set_xlim(-0.2, 1.0)
ax.set_title(f"Fig 3 — cos(δ, Δ_true) by ridge weight ({MODEL_LABEL}, n=1)")
ax.legend(loc="lower right"); style_ax(ax)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_cosine.png", dpi=150); plt.show()

ORDER = ["Unsteered"] + [f"Input Grad (n=1, λ={lam})" for lam in LAMBDAS] + ["Render write @1 (oracle)"]
hdr = ("| arm | Edit Index (−1…+1) | edit-frame RMSE | target RMSE | ghost RMSE | collateral RMSE "
       "| GT-traj RMSE | fidelity ratio |\n|---|---|---|---|---|---|---|---|\n")
rows = []
for name in ORDER:
    c = CARDS[name]
    rows.append(f"| {name} | {c['edit_index']:+.2f} | {c['edit_frame_rmse']:.3f} | {c['target_rmse']:.3f} "
                f"| {c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} "
                f"| {fidelity_ratio(c, CARDS['Unsteered']):.2f} |")
display(Markdown("**§4 scorecard** (canonical set; RMSE in intensity units vs the clean edited-world render; "
                 "step 0 = frame ef)\n\n" + hdr + "\n".join(rows)))

FIG4 = ["Unsteered", f"Input Grad (n=1, λ=0.0)", f"Input Grad (n=1, λ={LAM_MAIN})", "Render write @1 (oracle)"]
colors = ["#999999", "#56B4E9", "#0072B2", "#009E73"]
fig, ax = plt.subplots(figsize=(8, 4))
for name, col in zip(FIG4, colors):
    ax.plot(CARDS[name]["edit_index_by_step"], color=col, lw=1.8, marker="o", ms=3.5, label=name)
ax.axhline(0, color="#555555", lw=0.8, ls=":")
ax.set_xlabel("rollout step (0 = edit frame ef)"); ax.set_ylabel("Edit Index (−1…+1)")
ax.set_ylim(-1.05, 1.05)
ax.set_title(f"Fig 4 — Edit Index by rollout step ({MODEL_LABEL})")
ax.legend(fontsize=8, loc="upper right"); style_ax(ax)
fig.tight_layout(); fig.savefig(f"{OUT}/fig4_index_by_step.png", dpi=150); plt.show()

In [ ]:
# [6] Fig 5 — observation-space waterfall (canonical fixed spec; one helper).
N_CTX = 6
ctx_obs = edits.obs[:N, ef - N_CTX:ef, :].astype(np.float32)
DARK, TXT, TICK, EDIT_C = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850"
TARGET_C, GHOST_C = "#00E676", "#FF5252"
def _cx(m):
    i = np.where(m)[0]; return i.mean() if i.size else np.nan
tgt_cx = np.array([_cx(ZONES.target[i]) for i in range(N)])
pre_cx = np.array([_cx(ZONES.ghost[i]) for i in range(N)])

def waterfall_grid(col_titles, col_bodies, samples, suptitle, fname):
    """col_bodies[c]: (N, L, R) rows BELOW the N_CTX context frames; every column its OWN free-run
    from step 0 (= frame ef). No shared teacher-forced ef row (banned; see CLAUDE.md)."""
    ncol = len(col_titles)
    fig, axes = plt.subplots(len(samples), ncol, figsize=(3.0 * ncol, 3.4 * len(samples)),
                             squeeze=False, facecolor=DARK)
    for r, smp in enumerate(samples):
        for c in range(ncol):
            ax = axes[r][c]; ax.set_facecolor(DARK)
            panel = np.clip(np.concatenate([ctx_obs[smp], col_bodies[c][smp]], axis=0), 0, 1)
            ax.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1,
                      interpolation="nearest")
            for sp in ax.spines.values(): sp.set_edgecolor(TICK)
            ax.axhline(N_CTX - 0.5, color=EDIT_C, lw=1.4, ls="--", alpha=0.95)
            if not np.isnan(tgt_cx[smp]): ax.axvline(tgt_cx[smp], color=TARGET_C, lw=1.6, alpha=0.9)
            if not np.isnan(pre_cx[smp]): ax.axvline(pre_cx[smp], color=GHOST_C, ls="--", lw=1.6, alpha=0.9)
            if r == 0: ax.set_title(col_titles[c], fontsize=8, color=TXT)
            if c == 0:
                ax.set_ylabel(f"sample {smp} (teleport {teleport[smp]:.1f})\nsim frame", fontsize=8, color=TXT)
                ax.set_yticks([0, N_CTX, N_CTX + 7, N_CTX + 14])
                ax.set_yticklabels([ef - N_CTX, ef, ef + 7, ef + 14], fontsize=7)
            else: ax.set_yticks([])
            ax.set_xlabel("ray", fontsize=8, color=TXT); ax.tick_params(colors=TICK, labelsize=7)
    handles = [Line2D([0], [0], color=TARGET_C, lw=2.2, label="object target location"),
               Line2D([0], [0], color=GHOST_C, ls="--", lw=2.2, label="ghost (pre-edit) location"),
               Line2D([0], [0], color=EDIT_C, ls="--", lw=2.2,
                      label=f"edit applied here ({N_CTX} noisy context frames above; every row below is that "
                            f"column's OWN free-run, step 0 = frame {ef})")]
    fig.legend(handles=handles, loc="upper center", ncol=2, fontsize=8.5, frameon=False,
               labelcolor=TXT, bbox_to_anchor=(0.5, 0.965))
    fig.suptitle(suptitle, y=1.0, fontsize=10.5, color=TXT)
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(fname, dpi=150, facecolor=DARK, bbox_inches="tight"); plt.show()

WF = ["Unsteered", "Input Grad (n=1, λ=0.0)", f"Input Grad (n=1, λ={LAM_MAIN})", "Render write @1 (oracle)"]
titles = ["GT (sim clean obs)"] + [f"{n}\nEdit Index {CARDS[n]['edit_index']:+.2f}" for n in WF]
bodies = [gt_roll] + [ROLLS[n] for n in WF]
waterfall_grid(titles, bodies, SAMPLES,
               f"Fig 5 — free-run waterfalls: input-gradient steering arms vs references ({MODEL_LABEL})",
               f"{OUT}/fig5_waterfall.png")

## Current results (updated 2026-08-11)

- **The probe is mostly driven** (residual 3.45 → 0.25–0.42 for λ ≤ 0.1; only λ=1.0 stalls at 1.68) — harder than
  on the transformer (0.06–0.19), consistent with the write having to pass through one recurrent update.
- **The perturbation is even less aligned than the transformer's**: cos(δ*, Δ_true) ≈ +0.09…+0.13 (82–85°),
  first gradient +0.049 (87°); chance −0.045. At λ=0, ‖δ‖ = 4.17 **exceeds** the true edit's norm (3.45) while
  achieving almost nothing semantically — Fig 2's λ=0 column is saturated binary fuzz.
- **Generation response**: Edit Index −0.68 → −0.44…−0.57; fidelity ≈ 1.0 (ignored, not destroyed). The Fig 5
  waterfalls show a faint secondary streak appearing near the target in some samples — a partial write — while
  the ghost object persists untouched.
- **Same surface, oracle content: −0.01** (Render write @1) — matches the registry's First Obs TF (−0.08):
  one frame of even *perfect* evidence only neutralizes the GRU's belief, it does not relocate it. Note the
  transformer's oracle arm reaches +0.27 on the same content: the buffer write is a stronger channel than one
  recurrent update.

## Summary (interpretation — clearly marked as such)

Replicates the transformer notebook's verdict with a weaker write channel: the probe gradient finds an
off-manifold direction that flips the readout without looking like the edited world, and the recurrence filters
it out almost entirely. The GRU adds a second bottleneck the transformer doesn't have — even oracle-rendered
content is inertia-capped at ≈ 0 after one frame. So for the GRU, input-gradient steering fails twice over:
wrong direction (adversarial), and a channel that needs sustained evidence (cf. Freeze-time Interp. TF, which
wins by supplying N≈3–8 rendered frames).